## Máster en Big Data y Data Science

### Metodologías de gestión y diseño de proyectos de big data

#### AP1 - Limpieza de los datos

---

En esta libreta se realiza una evaluación básica de calidad de los datos del escenario y se ejecutan acciones de limpieza.

In [28]:
# Se importan las librerías necesarias y se suprimen las advertencias
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore',category=FutureWarning)
warnings.filterwarnings('ignore',category=UserWarning)

Lectura del mismo dataset inicial

In [29]:
# Lectura de los datos
df_creditos = pd.read_csv("../data/raw/datos_creditos.csv", sep=";")
display(df_creditos.head(5))

df_tarjetas = pd.read_csv("../data/raw/datos_tarjetas.csv", sep=";")
display(df_tarjetas.head(5))

,id_cliente,edad,importe_solicitado,duracion_credito,antiguedad_empleado,situacion_vivienda,ingresos,objetivo_credito,pct_ingreso,tasa_interes,estado_credito,falta_pago
0,713061558.0,22,35000,3,123.0,ALQUILER,59000,PERSONAL,0.59,16.02,1,Y
1,768805383.0,21,1000,2,5.0,PROPIA,9600,EDUCACIÓN,0.10,11.14,0,N
2,818770008.0,25,5500,3,1.0,HIPOTECA,9600,SALUD,0.57,12.87,1,N
3,713982108.0,23,35000,2,4.0,ALQUILER,65500,SALUD,0.53,15.23,1,N
4,710821833.0,24,35000,4,8.0,ALQUILER,54400,SALUD,0.55,14.27,1,Y


,id_cliente,antiguedad_cliente,estado_civil,estado_cliente,gastos_ult_12m,genero,limite_credito_tc,nivel_educativo,nivel_tarjeta,operaciones_ult_12m,personas_a_cargo
0,713061558.0,36.0,CASADO,ACTIVO,1088.0,M,4010.0,UNIVERSITARIO_COMPLETO,Blue,24.0,2.0
1,768805383.0,39.0,CASADO,ACTIVO,1144.0,M,12691.0,SECUNDARIO_COMPLETO,Blue,42.0,3.0
2,818770008.0,44.0,SOLTERO,ACTIVO,1291.0,F,8256.0,UNIVERSITARIO_COMPLETO,Blue,33.0,5.0
3,713982108.0,36.0,CASADO,ACTIVO,1887.0,M,3418.0,UNIVERSITARIO_COMPLETO,Blue,20.0,3.0
4,710821833.0,54.0,CASADO,ACTIVO,1314.0,M,9095.0,DESCONOCIDO,Blue,26.0,1.0


##### Transformaciones sobre los datos

Verificación de nulos

In [30]:
# Se verifica la cantidad de valores nulos en cada columna
print("Valores nulos en el dataset de créditos:")
for col in df_creditos.columns:
   print(f"Atributo: {col} - Valores nulos: {df_creditos[col].isna().sum()} ({df_creditos[col].isna().sum()/len(df_creditos)*100:.2f}%)")

Valores nulos en el dataset de créditos:
Atributo: id_cliente - Valores nulos: 0 (0.00%)
Atributo: edad - Valores nulos: 0 (0.00%)
Atributo: importe_solicitado - Valores nulos: 0 (0.00%)
Atributo: duracion_credito - Valores nulos: 0 (0.00%)
Atributo: antiguedad_empleado - Valores nulos: 337 (3.33%)
Atributo: situacion_vivienda - Valores nulos: 0 (0.00%)
Atributo: ingresos - Valores nulos: 0 (0.00%)
Atributo: objetivo_credito - Valores nulos: 0 (0.00%)
Atributo: pct_ingreso - Valores nulos: 0 (0.00%)
Atributo: tasa_interes - Valores nulos: 912 (9.01%)
Atributo: estado_credito - Valores nulos: 0 (0.00%)
Atributo: falta_pago - Valores nulos: 0 (0.00%)


In [31]:
# Se verifica la cantidad de valores nulos en cada columna
print("Valores nulos en el dataset de tarjetas:")
for col in df_tarjetas.columns:
   print(f"Atributo: {col} - Valores nulos: {df_tarjetas[col].isna().sum()} ({df_tarjetas[col].isna().sum()/len(df_tarjetas)*100:.2f}%)")

Valores nulos en el dataset de tarjetas:
Atributo: id_cliente - Valores nulos: 0 (0.00%)
Atributo: antiguedad_cliente - Valores nulos: 0 (0.00%)
Atributo: estado_civil - Valores nulos: 0 (0.00%)
Atributo: estado_cliente - Valores nulos: 0 (0.00%)
Atributo: gastos_ult_12m - Valores nulos: 0 (0.00%)
Atributo: genero - Valores nulos: 0 (0.00%)
Atributo: limite_credito_tc - Valores nulos: 0 (0.00%)
Atributo: nivel_educativo - Valores nulos: 0 (0.00%)
Atributo: nivel_tarjeta - Valores nulos: 0 (0.00%)
Atributo: operaciones_ult_12m - Valores nulos: 0 (0.00%)
Atributo: personas_a_cargo - Valores nulos: 0 (0.00%)


Selección de datos (filtro de columnas)

In [32]:
# Se establece qué columnas se eliminan
col_eliminar_creditos = []
col_eliminar_tarjetas = ['nivel_tarjeta']

# Se ejecuta la operación
df_creditos.drop(col_eliminar_creditos, inplace=True, axis=1)
df_tarjetas.drop(col_eliminar_tarjetas, inplace=True, axis=1)

Limpieza de datos (filtro de filas)

**Modificacion**
1. Se decarta la eliminacion directa de los valores nulos con la finalidad de procesarlos y realizar la imputacion mas adecuada al momento: Mediana.

In [33]:
# Checkpoint
df_creditos_filtrado = df_creditos.copy()

# Se filtran los datos para eliminar registros con edades superiores a 90 años
df_creditos_filtrado = df_creditos_filtrado[df_creditos_filtrado['edad'] < 90]

# Tratamiento de valores nulos
## Eliminar todo resgitro con algun valor nulo (deprecado. Ver Modificacion 1).
#df_creditos_filtrado.dropna(inplace=True)
## Tratamiento de valores nulos para tasa_interes, se reemplaza por la media de la columna, agrupada  por objetivo_credito
df_creditos_filtrado['tasa_interes'] = df_creditos_filtrado.groupby('objetivo_credito')['tasa_interes'].transform(lambda x: x.fillna(x.mean()))
## Tratamiento de nulos para antiguedad_empleado, se reemplaza por la mediana de la columna, agrupada por edad
df_creditos_filtrado['antiguedad_empleado'] = df_creditos_filtrado.groupby('edad')['antiguedad_empleado'].transform(lambda x: x.fillna(x.median()))

Integración de ambos datasets en uno solo

In [34]:
df_integrado = pd.merge(df_creditos_filtrado, df_tarjetas, on='id_cliente', how='inner')

print(f"Filas del dataset integrado con los filtros realizados: {df_integrado.shape[0]}")
print(f"Columnas del dataset integrado post integración: {df_integrado.shape[1]}")
print(f'El dataset original tenía {df_creditos.shape[0]} filas en Créditos y {df_tarjetas.shape[0]} filas en Tarjetas.')
print(f'Se han eliminado {df_creditos.shape[0] - df_creditos_filtrado.shape[0]} filas en total.')
display(df_integrado.head(5))

Filas del dataset integrado con los filtros realizados: 10123
Columnas del dataset integrado post integración: 21
El dataset original tenía 10127 filas en Créditos y 10127 filas en Tarjetas.
Se han eliminado 4 filas en total.


,id_cliente,edad,importe_solicitado,duracion_credito,antiguedad_empleado,situacion_vivienda,ingresos,objetivo_credito,pct_ingreso,tasa_interes,...,falta_pago,antiguedad_cliente,estado_civil,estado_cliente,gastos_ult_12m,genero,limite_credito_tc,nivel_educativo,operaciones_ult_12m,personas_a_cargo
0,713061558.0,22,35000,3,123.0,ALQUILER,59000,PERSONAL,0.59,16.02,...,Y,36.0,CASADO,ACTIVO,1088.0,M,4010.0,UNIVERSITARIO_COMPLETO,24.0,2.0
1,768805383.0,21,1000,2,5.0,PROPIA,9600,EDUCACIÓN,0.10,11.14,...,N,39.0,CASADO,ACTIVO,1144.0,M,12691.0,SECUNDARIO_COMPLETO,42.0,3.0
2,818770008.0,25,5500,3,1.0,HIPOTECA,9600,SALUD,0.57,12.87,...,N,44.0,SOLTERO,ACTIVO,1291.0,F,8256.0,UNIVERSITARIO_COMPLETO,33.0,5.0
3,713982108.0,23,35000,2,4.0,ALQUILER,65500,SALUD,0.53,15.23,...,N,36.0,CASADO,ACTIVO,1887.0,M,3418.0,UNIVERSITARIO_COMPLETO,20.0,3.0
4,710821833.0,24,35000,4,8.0,ALQUILER,54400,SALUD,0.55,14.27,...,Y,54.0,CASADO,ACTIVO,1314.0,M,9095.0,DESCONOCIDO,26.0,1.0


Generacion de atributos

En esta seccion se realiza la creacion de atributos derivados, de interes para el negocio.

In [ ]:
# ############## (SECCION RETO PROPUESTO) ################

# Atributo generado 1: capacidad de pago = importe solcitado / ingresos
df_integrado['capacidad_pago'] = df_integrado['importe_solicitado'] / df_integrado['ingresos']

# Atributo generado 2: ratio de endeudamiento = importe solicitado / (ingresos + importe solicitado)
df_integrado['ratio_endeudamiento'] = df_integrado['importe_solicitado'] / (df_integrado['ingresos'] + df_integrado['importe_solicitado'])

# Atributo generado 3: operaciones mensuales promedio = operaciones totales / 12 meses
df_integrado['operaciones_mensuales'] = df_integrado['operaciones_ult_12m'] / 12

# Atributo generado 4: gasto por transacción promedio = gasto total / operaciones totales
df_integrado['gasto_transaccion_promedio'] = df_integrado['gastos_ult_12m'] / df_integrado['operaciones_ult_12m']

# Atributo generado 5: Presion financiera
df_integrado['presion_financiera'] = (
    (df_integrado['gastos_ult_12m'] / 12 + df_integrado['importe_solicitado'] / (df_integrado['duracion_credito'] ) * 12)
    / (df_integrado['ingresos'] / 12)
)

# Atributo generado 6: Antiguedad relativa del empleado = antiguedad_empleado / edad
df_integrado['antiguedad_relativa'] = df_integrado['antiguedad_empleado'] / df_integrado['edad']

Transformaciones sobre atributos

**Moficacion**
1. Se descarta el mapping de estado civil
2. Se decarta el mapping de estado de credito
3. Se decarta la transformacion de Antinuedad del empleado
3. Se decarta la transformacion de edad

In [36]:
# Columna: estado_civil
#cambios_estado_civil = {
#    'CASADO' : 'C',
#    'SOLTERO' : 'S',
#    'DESCONOCIDO' : 'N',
#    'DIVORCIADO' : 'D',
#}
#
## Columna: estado_credito
#cambios_estado_credito = {
#    0: 'P',
#    1 : 'C',
#}
#
#estado_civil_N = df_integrado.loc[:, ('estado_civil')].map(cambios_estado_civil).rename('estado_civil_N')
#estado_credito_N = df_integrado.loc[:, ('estado_credito')].map(cambios_estado_credito).rename('estado_credito_N')

In [37]:
# Antiguedad del empleado
#etiquetas_a_e = ['menor_5', '5_a_10', 'mayor_10']
#rangos_a_e = [0, 4, 10, 50]
#valor_para_nan = 'NA'
#antiguedad_empleados_N = pd.cut(df_integrado['antiguedad_empleado'], 
#                                bins=rangos_a_e, 
#                                labels=etiquetas_a_e,
#                                right=False).cat.add_categories(valor_para_nan).fillna(valor_para_nan)
#
#display(antiguedad_empleados_N.value_counts())

# edad
#etiquetas_e = ['menor_25', '25_a_30']
#rangos_e = [0, 24, 50]
#edad_N = pd.cut(df_integrado['edad'], 
#                                bins=rangos_e, 
#                                labels=etiquetas_e)
#
#display(edad_N.value_counts())

In [38]:
# Se eliminan las columnas originales y se integran las nuevas columnas procesadas
columnas_a_eliminar = [
    #'estado_civil', 'estado_credito', 'antiguedad_empleado', 'edad',
    'id_cliente',
    'importe_solicitado',
    'operaciones_ult_12m',
    'gastos_ult_12m',
    'antiguedad_empleado',
    'edad',
]
df_integrado.drop(columnas_a_eliminar, inplace=True, axis=1)
#df_integrado = pd.concat([df_integrado, estado_civil_N, estado_credito_N, antiguedad_empleados_N, edad_N], axis=1)
#display(df_integrado.head(5))

Validaciones finales

In [39]:
# Se verifica la cantidad de valores nulos en cada columna del dataset integrado
print("Valores nulos en el dataset integrado:")
for col in df_integrado.columns:
   print(f"Atributo: {col} - Valores nulos: {df_integrado[col].isna().sum()} ({df_integrado[col].isna().sum()/len(df_integrado)*100:.2f}%)")

# Se verifica la presencia de valores duplicados en el dataset integrado
num_duplicados = df_integrado.duplicated().sum()
print(f"\nNúmero de filas duplicadas en el dataset integrado: {num_duplicados}")

Valores nulos en el dataset integrado:
Atributo: duracion_credito - Valores nulos: 0 (0.00%)
Atributo: situacion_vivienda - Valores nulos: 0 (0.00%)
Atributo: ingresos - Valores nulos: 0 (0.00%)
Atributo: objetivo_credito - Valores nulos: 0 (0.00%)
Atributo: pct_ingreso - Valores nulos: 0 (0.00%)
Atributo: tasa_interes - Valores nulos: 0 (0.00%)
Atributo: estado_credito - Valores nulos: 0 (0.00%)
Atributo: falta_pago - Valores nulos: 0 (0.00%)
Atributo: antiguedad_cliente - Valores nulos: 0 (0.00%)
Atributo: estado_civil - Valores nulos: 0 (0.00%)
Atributo: estado_cliente - Valores nulos: 0 (0.00%)
Atributo: genero - Valores nulos: 0 (0.00%)
Atributo: limite_credito_tc - Valores nulos: 0 (0.00%)
Atributo: nivel_educativo - Valores nulos: 0 (0.00%)
Atributo: personas_a_cargo - Valores nulos: 0 (0.00%)
Atributo: capacidad_pago - Valores nulos: 0 (0.00%)
Atributo: ratio_endeudamiento - Valores nulos: 0 (0.00%)
Atributo: operaciones_mensuales - Valores nulos: 0 (0.00%)
Atributo: gasto_tran

In [40]:
resumen = pd.DataFrame({
    'Atributo': df_integrado.columns,
    'Tipo de dato': df_integrado.dtypes,
    'Valores unicos': df_integrado.nunique(),
    'Valores nulos': df_integrado.isna().sum(),
    'Porcentaje nulos': df_integrado.isna().sum() / len(df_integrado) * 100
})

resumen

,Atributo,Tipo de dato,Valores unicos,Valores nulos,Porcentaje nulos
duracion_credito,duracion_credito,int64,3,0,0.0
situacion_vivienda,situacion_vivienda,object,4,0,0.0
ingresos,ingresos,int64,1821,0,0.0
objetivo_credito,objetivo_credito,object,6,0,0.0
pct_ingreso,pct_ingreso,float64,72,0,0.0
tasa_interes,tasa_interes,float64,312,0,0.0
estado_credito,estado_credito,int64,2,0,0.0
falta_pago,falta_pago,object,2,0,0.0
antiguedad_cliente,antiguedad_cliente,float64,44,0,0.0
estado_civil,estado_civil,object,4,0,0.0


----

Exportación del dataset unificado a un archivo nuevo (en el directorio de procesados)

In [41]:
# Exportar el DataFrame limpio a un nuevo archivo CSV
df_integrado.to_csv('../data/processed/datos_integrados.csv', index=False)